
### **Q1. What is the mathematical formula for a linear SVM?**
The decision boundary for a **linear SVM** is given by the equation:

\[
w^T x + b = 0
\]

where:
- \( w \) is the weight vector,
- \( x \) is the input feature vector,
- \( b \) is the bias term.

---

### **Q2. What is the objective function of a linear SVM?**
The optimization problem for **hard-margin SVM** (without slack variables) is:

\[
\min_{w, b} \frac{1}{2} \|w\|^2
\]

subject to the constraint:

$$
y_i (w^T x_i + b) \geq 1, \quad \forall i$$

For **soft-margin SVM** (allowing some misclassification):

$$
\min_{w, b, \xi} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^{n} \xi_i
$$

subject to:

$$
y_i (w^T x_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0, \quad \forall i
$$

where:
- \( C \) is a hyperparameter controlling the trade-off between maximizing the margin and minimizing classification error,
$$  \xi_i$$
 are slack variables allowing some misclassification.

---

### **Q3. What is the kernel trick in SVM?**
The **kernel trick** allows SVM to operate in a **higher-dimensional space** without explicitly transforming the data. Instead of computing the dot product in high-dimensional space, we use a kernel function:

$$
K(x_i, x_j) = \phi(x_i)^T \phi(x_j)
$$


Common kernel functions:
1.$$ **Linear Kernel:**  K(x_i, x_j) = x_i^T x_j  $$
2.$$ **Polynomial Kernel:**  K(x_i, x_j) = (x_i^T x_j + c)^d  $$
3.$$ **RBF Kernel:**  K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)   $$
4. $$ **Sigmoid Kernel:**  K(x_i, x_j) = \tanh(\alpha x_i^T x_j + c)   $$

This trick allows SVMs to handle non-linearly separable data efficiently.

---

### **Q4. What is the role of support vectors in SVM? Explain with an example.**
Support vectors are the **data points that lie closest to the decision boundary** and define the margin.

Example:
- Given two classes of points (e.g., cats vs. dogs), the **support vectors** are the points that are nearest to the separating hyperplane.
- The decision boundary is determined by these points rather than all the training data, making SVM efficient.

---

### **Q5. Illustrate with examples and graphs of Hyperplane, Marginal plane, Soft margin, and Hard margin in SVM.**
To visualize:
1. **Hyperplane:** The decision boundary separating classes.
2. **Marginal plane:** The two parallel boundaries around the hyperplane.
3. **Hard Margin SVM:** No misclassification allowed (for linearly separable data).
4. **Soft Margin SVM:** Allows some misclassification for better generalization.

I'll provide code to generate these visualizations below.

---

### **Q6. SVM Implementation through Iris dataset**
```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from mlxtend.plotting import plot_decision_regions

# Load the dataset
iris = datasets.load_iris()
X = iris.data[:, :2]  # Using first two features for visualization
y = iris.target

# Convert multi-class to binary (for simplicity)
y = (y != 0).astype(int)  # Classify class 1 vs rest

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the SVM model
svm_model = SVC(kernel='linear', C=1.0)
svm_model.fit(X_train, y_train)

# Predictions and accuracy
y_pred = svm_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Plot decision boundary
plt.figure(figsize=(8, 6))
plot_decision_regions(X_train, y_train, clf=svm_model, legend=2)
plt.title("SVM Decision Boundary on Iris Dataset")
plt.show()
```
- Try **different C values** (`C=0.1`, `C=10`) to observe the effect on decision boundary.

---

### **Bonus Task: Implementing Linear SVM from Scratch**
```python
from cvxopt import matrix, solvers
import numpy as np

class LinearSVM:
    def __init__(self, C=1.0):
        self.C = C

    def fit(self, X, y):
        n_samples, n_features = X.shape
        y = y.reshape(-1, 1) * 1.0

        # Quadratic programming setup
        K = np.dot(y * X, (y * X).T)
        P = matrix(K)
        q = matrix(-np.ones((n_samples, 1)))
        G = matrix(np.vstack((-np.eye(n_samples), np.eye(n_samples))))
        h = matrix(np.hstack((np.zeros(n_samples), np.ones(n_samples) * self.C)))
        A = matrix(y.T, (1, n_samples), 'd')
        b = matrix(0.0)

        # Solve quadratic programming problem
        sol = solvers.qp(P, q, G, h, A, b)
        alpha = np.array(sol['x'])

        # Get support vectors
        support_vectors = (alpha > 1e-5).flatten()
        self.w = np.sum(alpha * y * X, axis=0)
        self.b = np.mean(y[support_vectors] - np.dot(X[support_vectors], self.w))

    def predict(self, X):
        return np.sign(np.dot(X, self.w) + self.b)

# Training on the Iris dataset
model = LinearSVM(C=1.0)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Accuracy
accuracy = np.mean(predictions == y_test)
print("SVM from scratch Accuracy:", accuracy)
```
- This code solves the **quadratic optimization problem** manually and compares results with `sklearn` implementation.

---

## **Additional Questions**

### **Q1. Relationship between Polynomial and Kernel Functions**
A **polynomial function** is used in SVM as a **kernel function**:

$$
K(x_i, x_j) = (x_i^T x_j + c)^d
$$

where:
- \( d \) is the polynomial degree,
- \( c \) is a constant.

It allows SVM to **learn curved decision boundaries**.

---

### **Q2. Implementing SVM with Polynomial Kernel in Python**
```python
svm_poly = SVC(kernel='poly', degree=3, C=1.0)
svm_poly.fit(X_train, y_train)
print("Polynomial Kernel Accuracy:", accuracy_score(y_test, svm_poly.predict(X_test)))
```

---

### **Q3. Effect of Increasing \( \epsilon \) in SVR**
- **Larger \( \epsilon \):** More tolerance for error → fewer support vectors.
- **Smaller \( \epsilon \):** Less tolerance → more support vectors.

---

### **Q4. Effect of Parameters in SVR**
1. **Kernel Function:** Defines feature transformation (e.g., linear, RBF).
2. **C Parameter:** Higher \( C \) → lower margin, less misclassification.
3. **Epsilon \( \epsilon \):** Controls error tolerance.
4. **Gamma \( \gamma \):** Controls influence of data points in RBF kernel.

Example:
- High \( C \) for strict classification.
- Low \( C \) for better generalization.

---

### **Q5. Assignment (SVC with GridSearch)**
Use `GridSearchCV` to tune hyperparameters and `joblib` to save the model.
